# Exploratory Analysis of OpenAlex Dataset and Networks

In [1]:
import networkx as nx
import numpy as np
import pandas as pd
import sklearn
import sentence_transformers
import json

## text as data on abstracts

In [2]:
works = pd.read_csv("api_test/data_2013_2016/works.csv")
print(f"Number of works: {len(works)}")
works.head()

Number of works: 800


,openalex_id,doi,title,abstract,publication_year,publication_date,type,language,cited_by_count,referenced_works_count,referenced_works,is_oa,oa_status,oa_url,source_name,primary_topic,primary_topic_id,authors,author_ids
0,https://openalex.org/W2130781952,https://doi.org/10.1002/adhm.201300113,A New Design and Application of Bioelastomers ...,Glaucoma drainage device (GDD) implantation is...,2013,2013-07-09,article,en,4,22,https://openalex.org/W194547286; https://opena...,False,closed,NaN,Advanced Healthcare Materials,Glaucoma and retinal disorders,https://openalex.org/T10250,Quang Minh Luong; Lei Shang; Marcus Ang; Jen F...,https://openalex.org/A5060459854; ; https://op...
1,https://openalex.org/W2768710832,NaN,Being An Effective Communicator In A Team,When it came to pick a subject for our researc...,2013,2013-01-01,article,en,0,0,NaN,False,closed,NaN,PDXScholar (Portland State University),Organizational Learning and Leadership,https://openalex.org/T13163,Ashish Mishra; Jared Omulindi; Philip D. Jaffé...,https://openalex.org/A5100660163; https://open...
2,https://openalex.org/W2726385890,NaN,Dynamic Time Slot Assignment Scheme for Pictur...,NaN,2013,2013-02-28,article,en,0,0,NaN,False,closed,NaN,IEICE Technical Report; IEICE Tech. Rep.,Energy Efficient Wireless Sensor Networks,https://openalex.org/T10080,Nagai Ryosuke; Sakata Shiro; Komuro Nobuyoshi,https://openalex.org/A5087433314; https://open...
3,https://openalex.org/W186702817,https://doi.org/10.17077/0003-4827.1745,This Wicked Rebellion: Wisconsin Civil War Sol...,in the modern era.Historians mostly ignored th...,2013,2013-10-01,article,en,2,0,NaN,True,bronze,https://pubs.lib.uiowa.edu/annals-of-iowa/arti...,The Annals of Iowa,American History and Culture,https://openalex.org/T13025,Patrick G. Bass,https://openalex.org/A5088691622
4,https://openalex.org/W2004782196,https://doi.org/10.1016/j.aop.2013.05.017,WKB analysis of relativistic Stern–Gerlach mea...,NaN,2013,2013-06-13,article,en,18,35,https://openalex.org/W225340292; https://opena...,True,green,https://arxiv.org/pdf/1208.6434,Annals of Physics,Quantum Mechanics and Non-Hermitian Physics,https://openalex.org/T11262,Matthew C. Palmer; Maki Takahashi; Hans F. Wes...,; ;


In [3]:
# Temporary backfill for CSV files created before primary taxonomy columns were added.
# This cell can be deleted after regenerating the data with the updated extract.py.

taxonomy_columns = [
    "primary_topic_score",
    "primary_subfield", "primary_subfield_id",
    "primary_field", "primary_field_id",
    "primary_domain", "primary_domain_id",
]
missing_taxonomy_columns = [column for column in taxonomy_columns if column not in works.columns]

if missing_taxonomy_columns:
    with open("api_test/data_2013_2016/works_raw.json", encoding="utf-8") as file:
        raw_works = json.load(file)["results"]

    taxonomy_rows = []
    for work in raw_works:
        topic = work.get("primary_topic") or {}
        subfield = topic.get("subfield") or {}
        field = topic.get("field") or {}
        domain = topic.get("domain") or {}
        taxonomy_rows.append({
            "openalex_id": work["id"],
            "primary_topic_score": topic.get("score"),
            "primary_subfield": subfield.get("display_name"),
            "primary_subfield_id": subfield.get("id"),
            "primary_field": field.get("display_name"),
            "primary_field_id": field.get("id"),
            "primary_domain": domain.get("display_name"),
            "primary_domain_id": domain.get("id"),
        })

    taxonomy = pd.DataFrame(taxonomy_rows).set_index("openalex_id")
    if not taxonomy.index.is_unique:
        raise ValueError("Duplicate OpenAlex IDs in works_raw.json")
    if set(works["openalex_id"]) != set(taxonomy.index):
        raise ValueError("works.csv and works_raw.json contain different OpenAlex IDs")

    original_ids = works["openalex_id"].reset_index(drop=True)
    works = works.join(taxonomy[missing_taxonomy_columns], on="openalex_id", validate="one_to_one")
    if not works["openalex_id"].reset_index(drop=True).equals(original_ids):
        raise ValueError("Taxonomy join changed work order")

print(f"Primary taxonomy available for {works['primary_field_id'].notna().sum()} of {len(works)} works")

Primary taxonomy available for 797 of 800 works


In [4]:
# remove rows with missing abstracts
works = works.dropna(subset=["abstract"])

# remove rows with tiny abstracts
works = works[works["abstract"].str.len() > 20]
print(f"Number of works after cleaning: {len(works)}")

Number of works after cleaning: 502


### Count Vectorizer

In [5]:
from sklearn.feature_extraction.text import CountVectorizer

CountVectorizer = CountVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=5, # words should appear in at least 5 abstracts
    max_df=0.95
)

CV_X = CountVectorizer.fit_transform(works["abstract"]) # fit learns parameters, transform applies them, fit_transform does both

In [6]:
CV_X # number of unique words that appear in at least 5 abstracts)

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 20133 stored elements and shape (502, 1666)>

In [7]:
first_vector = CV_X[0].toarray() # first abstract's bag-of-words representation
print(f"Number of dimensions: {len(first_vector[0])}")
first_vector[0][:20] # first 20 dimensions of the first abstract's bag-of-words representation

Number of dimensions: 1666


array([0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0])

In [8]:
feature_names = CountVectorizer.get_feature_names_out() # exact words that appear in >= 2 abstracts
feature_names[200:220] # some examples

array(['banking', 'barriers', 'based', 'baseline', 'basic', 'basis',
       'beam', 'began', 'behavior', 'behaviors', 'behaviour', 'believed',
       'benefit', 'best', 'better', 'binding', 'biological', 'biomass',
       'birth', 'black'], dtype=object)

### TF-IDF
A weighted CountVectorizer where common terms have lower weights and rare words are weighted higher

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer

TfidfVectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    min_df=5, 
    max_df=0.95
)
TFIDF_X = TfidfVectorizer.fit_transform(works["abstract"]) 

In [10]:
TFIDF_X

<Compressed Sparse Row sparse matrix of dtype 'float64'
	with 20133 stored elements and shape (502, 1666)>

In [11]:
print(f"Number of dimensions: {len(TFIDF_X[0].toarray()[0])}")
TFIDF_X[0].toarray()[0][:20] # first 20 dimensions of the first abstract's TF-IDF representation

Number of dimensions: 1666


array([0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.1053751, 0.       , 0.       ,
       0.       , 0.       , 0.       , 0.       , 0.       , 0.       ,
       0.       , 0.       ])

In [12]:
feature_names_tfidf = TfidfVectorizer.get_feature_names_out() # exact words that appear in >= 5 abstracts
feature_names_tfidf[200:220] # some examples (same as CountVectorizer)

# Check if all feature names are the same
print(f"Are TF-IDF features and CV features equal? {np.array_equal(feature_names_tfidf, feature_names)}")

Are TF-IDF features and CV features equal? True


In [13]:
# find the top 10 words with the highest TF-IDF scores for the first abstract
first_abstract_tfidf = TFIDF_X[0].toarray()[0]
top_10_indices = first_abstract_tfidf.argsort()[-10:][::-1] # indices of the top 10 words
top_10_words = feature_names_tfidf[top_10_indices]
top_10_scores = first_abstract_tfidf[top_10_indices]
print("Top 10 words with highest TF-IDF scores for the first abstract:")
for word, score in zip(top_10_words, top_10_scores):
    print(f"{word}: {score}")

Top 10 words with highest TF-IDF scores for the first abstract:
device: 0.4062126337331511
vivo: 0.3679900710043033
vitro: 0.3679900710043033
pressure: 0.22854234170773963
phase: 0.1939658177999604
risk: 0.18428065963523446
compared: 0.1640546897438951
minimize: 0.1354042112443837
polymer: 0.13155943280775942
devices: 0.13155943280775942


### Document embedding

In [14]:
from sentence_transformers import SentenceTransformer

embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = embedding_model.encode(works["abstract"].tolist(), show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/16 [00:00<?, ?it/s]

In [15]:
print(embeddings.shape) # rows(abstracts) x columns(embeddings)

# example of one embedding (384 latent features/dimensions)
print(embeddings[0][:10]) # first 10 dimensions of the first abstract's embedding

(502, 384)
[-0.02168146 -0.06114104  0.11256856 -0.07322948 -0.05123238 -0.02665479
  0.09974257  0.10423435 -0.01161763  0.01009952]


## Vector Representation techniques
- Similarity / distance
  - cosine similarity
  - Euclidean distance
  - nearest neighbors
  - useful for: “Which papers are most similar to this paper?”
- Clustering
  - K-means
  - hierarchical clustering
  - DBSCAN / HDBSCAN
  - useful for: discovering groups of semantically similar papers without predefined labels
- Dimensionality reduction
  - PCA
  - Truncated SVD, especially common for TF-IDF
  - UMAP
  - t-SNE
  - useful for projecting hundreds/thousands of dimensions down to 2D or 3D for visualization
- Classification
  - logistic regression
  - SVM
  - random forest, etc.
  - useful if you already have OpenAlex field labels and want to predict discipline from abstracts
- Regression
  - use the text vectors to predict a continuous outcome, such as citation count
  - for high-dimensional TF-IDF, regularized regression like Ridge/Lasso is often useful
- Nearest-neighbor search / recommendation
  - find the \(k\) closest abstracts to each article
  - essentially a paper recommendation system
- Network construction
  - papers = nodes
  - connect two papers if cosine similarity exceeds some threshold, or if one is among the other's \(k\) nearest neighbors
  - then do community detection, centrality, interdisciplinarity analysis, etc.
- Temporal analysis
  - calculate average embeddings or topic/cluster composition by year
  - measure how fields move closer together or farther apart in semantic space over time

### Cosine similarity

In [16]:
from sklearn.metrics.pairwise import cosine_similarity

cos_cv = cosine_similarity(CV_X)
cos_tfidf = cosine_similarity(TFIDF_X)
cos_embed = cosine_similarity(embeddings)

In [17]:
print(cos_cv.shape)
print(f"Cosine similarity between abstract 0 and 1 (CV): {cos_cv[0][1]}")
print(f"Cosine similarity between abstract 0 and 1 (TF-IDF): {cos_tfidf[0][1]}")
print(f"Cosine similarity between abstract 0 and 1 (Embeddings): {cos_embed[0][1]}")

(502, 502)
Cosine similarity between abstract 0 and 1 (CV): 0.015765635809529454
Cosine similarity between abstract 0 and 1 (TF-IDF): 0.014295390329820393
Cosine similarity between abstract 0 and 1 (Embeddings): -0.08616819977760315


In [18]:
# find the top 5 most similar abstracts to the first abstract
def top_neighbors(sim_matrix, i, k=5):
    return np.argsort(sim_matrix[i])[::-1][1:k+1]

print(top_neighbors(cos_cv, 0))
print(top_neighbors(cos_tfidf, 0))
print(top_neighbors(cos_embed, 0))

[407 125  88 275 408]
[ 88 408 125 475 144]
[408 462 364  62 305]


In [19]:
# number of common top neighbors
def neighbor_overlap(sim_matrix1, sim_matrix2, k=5):
    """
    output = list of abstracts and its number of common neighbors in k total neighbors
    """
    overlaps = []

    for i in range(sim_matrix1.shape[0]):
        neighbors1 = set(top_neighbors(sim_matrix1, i, k))
        neighbors2 = set(top_neighbors(sim_matrix2, i, k))

        overlap = len(neighbors1 & neighbors2)
        overlaps.append(overlap)

        average_overlap_prop = (sum(overlaps)/len(overlaps))/k

    return average_overlap_prop, overlaps

In [20]:
cv_tfidf = neighbor_overlap(cos_cv, cos_tfidf, 10)
cv_embed = neighbor_overlap(cos_cv, cos_embed, 10)
tfidf_embed = neighbor_overlap(cos_tfidf, cos_embed, 10)
print(f"CV and TF-IDF: {cv_tfidf}")
print(f"CV and embedder: {cv_embed}")
print(f"TF-IDF and embedder: {tfidf_embed}")

CV and TF-IDF: (0.7394422310756972, [4, 9, 8, 7, 6, 5, 8, 6, 9, 6, 10, 8, 9, 7, 8, 5, 9, 9, 9, 9, 10, 7, 8, 9, 8, 9, 9, 9, 6, 8, 7, 6, 7, 9, 10, 9, 9, 8, 4, 6, 8, 8, 9, 7, 9, 8, 2, 6, 9, 6, 5, 5, 6, 8, 9, 6, 9, 7, 9, 9, 8, 4, 8, 9, 5, 6, 8, 7, 7, 9, 8, 8, 9, 5, 8, 6, 3, 8, 9, 5, 8, 7, 8, 6, 6, 7, 9, 8, 7, 9, 7, 6, 9, 8, 5, 7, 9, 5, 4, 9, 7, 10, 8, 9, 8, 9, 5, 7, 8, 7, 6, 9, 7, 9, 7, 6, 8, 8, 9, 8, 7, 7, 10, 7, 8, 9, 6, 7, 5, 5, 8, 5, 8, 6, 7, 8, 8, 8, 6, 6, 8, 9, 7, 9, 9, 8, 8, 6, 7, 7, 8, 4, 7, 9, 9, 7, 8, 7, 8, 10, 5, 5, 10, 8, 8, 7, 9, 8, 8, 8, 5, 6, 9, 8, 6, 7, 8, 6, 9, 6, 7, 6, 7, 8, 9, 7, 7, 6, 9, 8, 9, 5, 5, 9, 8, 9, 6, 7, 9, 8, 7, 10, 6, 9, 8, 7, 8, 7, 9, 6, 10, 7, 5, 7, 10, 5, 8, 8, 9, 9, 7, 5, 6, 8, 8, 8, 9, 6, 6, 8, 7, 9, 7, 9, 10, 6, 7, 8, 4, 9, 6, 8, 10, 8, 8, 8, 5, 6, 8, 5, 8, 5, 8, 8, 6, 10, 9, 5, 8, 7, 6, 7, 9, 7, 7, 8, 7, 7, 10, 7, 7, 8, 7, 8, 9, 9, 7, 8, 7, 9, 4, 8, 10, 7, 9, 8, 7, 8, 9, 8, 9, 5, 10, 9, 9, 8, 6, 8, 9, 7, 6, 7, 6, 8, 6, 8, 8, 7, 5, 6, 8, 10, 7, 6, 9, 8

Take the top 10 most similar (cosine similarity) abstracts for each 502 abstracts. For each abstract, compare how many abstracts the top 10 have in common between the two encoding methods. List out how many are in common and the average prop of common abstracts.

CV and TF-IDF produce similar results when measuring cosine similarity. The embedder vectors resulted in abstract groupings that are quite different based on cosine similarity.